In [113]:
import json
import os

with open("filter_information.json") as f:
    filter_list = json.load(f)

filter_name_to_id = dict([(i["label"].strip(), i["id"]) for i in filter_list])
good_keys = set(filter_name_to_id.keys())
filter_id_to_name = dict([(i["id"], i["label"].strip()) for i in filter_list])

def check_security_and_get_keys(text):
    
    just_ors = text.replace("AND","OR")
    
    def rem_extra_brackets(a):
        left = a.count("(")
        right = a.count(")")
        if left > right:
            return a[1:]
        elif right >left:
            return a[:-1]
        else:
            return a
    
    potential_keys = set([rem_extra_brackets(i.strip()) for i in just_ors.split("OR")])
    try:
        assert (set(potential_keys) - good_keys == set())
    except AssertionError:
        print([i for i in potential_keys if i not in good_keys])
    return list(potential_keys)

def convert_filters_to_filter_id_in_logic_expression(text, ids_to_studies):
    keys_list = check_security_and_get_keys(text)
    id_list = [filter_name_to_id[t] for t in keys_list]
    new_text = text    
    for i,t in enumerate(keys_list):
        t_id = id_list[i]
        studies = f"{set(ids_to_studies[t_id])}"
        new_text = new_text.replace(t, studies)
    study_expression =  new_text.replace("AND","&").replace("OR","|")
    return study_expression

In [121]:
text = """(8010/0 Epithelial tumor, benign OR 8010/6 Carcinoma, metastatic, NOS) OR (C00-C14 Lip, oral cavity and pharynx OR C00 Lip OR Acute myeloid leukaemia (AML)) AND (Material type OR Genetic material) AND (Material type OR Other Fluids - eg urine) AND (Cell Source OR Patient derived) AND Imaging Data"""

In [122]:
keys_list = check_security_and_get_keys(text)
id_list = [filter_name_to_id[t] for t in keys_list]

In [123]:
eval(convert_filters_to_filter_id_in_logic_expression(text, ids_to_studies))

{0,
 1,
 3,
 7,
 8,
 10,
 11,
 16,
 18,
 19,
 20,
 22,
 23,
 24,
 27,
 28,
 32,
 36,
 38,
 39,
 42,
 45,
 46,
 48}

In [97]:
[i for i in filter_name_to_id2.keys() if (i[0]==" " or i[-1]==" ")]

['8122/3 Urothelial carcinoma, sarcomatoid ',
 '8316/1 Multilocular cystic renal neoplasm of low malignant potential ',
 'Bloods ']

In [70]:
eval(studies)

{14, 46}

In [63]:
{1,2,3} & {1,2}

{1, 2}

In [ ]:
def convert_filters_to_studies_in_logic_expression(text):
    terms_list = text.replace("AND","OR").replace(")","").replace("(","").split(" OR ")
    
    id_list = [filter_name_to_id[t] for t in terms_list]
    
    new_text = text
    for i,t in enumerate(terms_list):
        studies = f"{set(ids_to_studies[id_list[i]])}"
        if t in new_text:
            new_text = new_text.replace(t, studies)
        else:
            t_with_brackets = filter_id_to_name[id_list[i]]
            new_text = new_text.replace(t_with_brackets, studies)
    return studies
    

In [47]:
convert_filters_to_studies_in_logic_expression(text)

'{2, 6, 10, 11, 12, 42, 14, 43, 44, 45, 46, 19, 47, 25, 27}'

In [44]:
f"{ids_to_studies[ids[3]]}"

'[0, 1, 13, 14, 16, 19, 20, 23, 28, 29, 34, 36, 38, 40, 46]'

In [111]:
import random

ids = list(set(id_list))
mock_study_ids = [sorted(list(set([random.randint(0,50) for i in range(random.randint(5,20))]))) for j in range(len(ids))]
ids_to_studies = dict(zip(ids, mock_study_ids))
ids_to_studies[ids[3]]

[14, 23, 31, 32, 42, 45]

In [112]:
ids_to_studies

{'0_0_0_0': [2, 7, 10, 12, 20, 27, 33, 36, 40, 41, 42],
 '0_0_0_0_0': [22, 27, 29, 34, 35, 43, 44, 45],
 '0_2_0_0': [3, 14, 19, 27, 32, 43, 47, 49, 50],
 '0_2_1_1_1': [14, 23, 31, 32, 42, 45],
 '0_0_1_1_0': [0, 3, 7, 11, 19, 20, 24, 28, 32, 38, 39, 42, 48],
 '0_2_0_0_0': [0, 1, 3, 10, 13, 25, 26, 30, 33, 43, 44, 47, 48, 49],
 '0_0_1_1_3': [1, 8, 10, 16, 18, 22, 23, 24, 32, 36, 42, 45, 46, 48],
 '0_2_3_2': [2, 4, 6, 10, 13, 14, 16, 26, 27, 28, 34, 40, 45, 46, 47],
 '0_0_2_3': [2, 6, 15, 21, 22, 23, 27, 29, 37, 43, 48, 49],
 '0_2_1_1': [2, 8, 27, 30, 43, 46, 47, 49],
 '0_2_0_0_3': [0, 4, 8, 9, 11, 14, 20, 28, 37, 46, 47, 49],
 '0_2_0_0_2': [1, 8, 12, 16, 18, 27, 28, 29, 31, 32, 34, 38, 47]}